# ViSEF — Kaggle T4 × 2 (seed 42 và 123)

Notebook này chạy benchmark multi-seed trên **hai GPU T4 đồng thời**: GPU 0 xử lý seed 42, GPU 1 xử lý seed 123. Mỗi seed chạy tuần tự bốn cấu hình; hai tiến trình seed độc lập. Seed 456 được chạy trong notebook Colab riêng `ViSEF_seed456_colab.ipynb`.

Cần bật Internet để clone mã nguồn và bật accelerator GPU T4 × 2. Gắn dataset Brain Tumor MRI vào Kaggle Notebook. Hai notebook dùng chung `partitions` được tạo một lần từ seed 42 để giữ partition giống nhau giữa các seed.

In [ ]:
# Kaggle đã cài sẵn PyTorch/CUDA. Cài thêm dependency còn thiếu.
!pip -q install opacus
import torch, os, sys, subprocess, json, time, shutil
from pathlib import Path
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Hãy bật accelerator T4 × 2 trong Kaggle Settings.'


In [ ]:
# Lấy mã nguồn và tìm dataset đã gắn ở /kaggle/input.
PROJECT = Path('/kaggle/working/KHKT')
if not (PROJECT / 'src/run_simulation.py').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/nguyenquocthanh1213/KHKT.git',str(PROJECT)], check=True)
os.chdir(PROJECT)

DATA_CANDIDATES = [Path('/kaggle/input/brain-tumor-mri-dataset')]
DATA_CANDIDATES += [p for p in Path('/kaggle/input').glob('*') if p.is_dir()]
DATA_ROOT = next((p for c in DATA_CANDIDATES for p in ([c] + list(c.glob('*'))) 
                  if (p/'Training').is_dir() and (p/'Testing').is_dir()), None)
assert DATA_ROOT is not None, 'Không tìm thấy Training/ và Testing/. Hãy gắn dataset Brain Tumor MRI vào notebook.'
DATA_ROOT = DATA_ROOT.resolve()
print('PROJECT:', PROJECT, '| DATA_ROOT:', DATA_ROOT)


In [ ]:
# Kiểm tra dữ liệu, rồi tạo partition đúng một lần để không có race giữa hai worker.
subprocess.run([sys.executable,'src/dedup_check.py','--data_root',str(DATA_ROOT),'--threshold','0'],check=True)
subprocess.run([sys.executable,'src/partition.py','--data_root',str(DATA_ROOT),'--out_dir','partitions',
                '--clients','A:glioma','B:meningioma','C:notumor','D:pituitary',
                '--samples_per_client','1200','--dominant_frac','0.7','--val_per_class','100','--seed','42'],check=True)


## Chạy hai seed song song

Mỗi tiến trình chỉ nhìn thấy một GPU vật lý qua `CUDA_VISIBLE_DEVICES`; trong tiến trình đó GPU sẽ có chỉ số logic `cuda:0`. Checkpoint và log được lưu trong workspace Kaggle. Có thể chạy lại cell để resume: cấu hình đã có `eval_test.json` sẽ bị bỏ qua.

In [ ]:
QUICK_MODE = False
SEEDS = [42, 123] if not QUICK_MODE else [42, 123]
ROUNDS = 20 if not QUICK_MODE else 5
EPOCHS, BATCH, LR = 3, 32, 1e-4
DOM_FRAC, TRAIN_MODE = 0.7, 'linear'
CONFIGS = {
    'fedavg': dict(mu=0.0, dp=False),
    'fedprox': dict(mu=0.01, dp=False),
    'fedprox_dp': dict(mu=0.01, dp=True),
    'fedprox_dp_autoclip': dict(mu=0.01, dp=True, auto_clip=True),
}

def run_seed(seed, gpu):
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    env['OMP_NUM_THREADS'] = '2'
    log_path = Path('runs') / f'kaggle_seed{seed}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8', buffering=1) as log:
        for name, cfg in CONFIGS.items():
            tag = f'p4_linear_{name}_seed{seed}'
            run_dir = Path('runs') / tag
            if (run_dir/'eval/eval_test.json').exists():
                print(f'[SKIP] {tag} done', file=log, flush=True)
                continue
            cmd = [sys.executable,'src/run_simulation.py','--data_root',str(DATA_ROOT),
                   '--tag',tag,'--partition_dir','partitions','--rounds',str(ROUNDS),
                   '--epochs',str(EPOCHS),'--batch_size',str(BATCH),'--lr',str(LR),
                   '--train_mode',TRAIN_MODE,'--mu',str(cfg['mu']),'--dominant_frac',str(DOM_FRAC),
                   '--seed',str(seed),'--skip_eval','--device','auto']
            if cfg['dp']:
                cmd += ['--dp','--noise_multiplier','1.0','--max_grad_norm','1.0','--verify_dp']
                if cfg.get('auto_clip'): cmd += ['--auto_clip']
            print(f'\n===== seed={seed}, gpu={gpu}, config={name} =====',file=log,flush=True)
            subprocess.run(cmd,env=env,stdout=log,stderr=subprocess.STDOUT,check=True)
            subprocess.run([sys.executable,'src/evaluate.py','--run_dir',str(run_dir),'--split','val',
                            '--partition_dir','partitions','--all_rounds','--device','auto','--no_plots'],
                           env=env,stdout=log,stderr=subprocess.STDOUT,check=True)
            val=json.loads((run_dir/'eval/eval_val.json').read_text(encoding='utf-8'))['results']
            srv=[r for r in val if r['kind']=='server' and r['round']>0]
            if not srv: raise RuntimeError(f'Không có validation server round: {tag}')
            best=max(srv,key=lambda r:r['macro_f1'])['round']
            subprocess.run([sys.executable,'src/evaluate.py','--run_dir',str(run_dir),'--split','test',
                            '--data_root',str(DATA_ROOT),'--server_rounds',str(best),'--client_rounds','',
                            '--device','auto','--no_plots'],env=env,stdout=log,stderr=subprocess.STDOUT,check=True)
        print(f'Hoàn tất seed {seed}; log: {log_path}',file=log,flush=True)

import threading, traceback
errors=[]
def guarded(seed,gpu):
    try: run_seed(seed,gpu)
    except Exception as e: errors.append((seed,repr(e))); traceback.print_exc()
threads=[threading.Thread(target=guarded,args=(seed,gpu),daemon=False) for seed,gpu in zip(SEEDS,[0,1])]
t0=time.time()
for t in threads: t.start()
for t in threads: t.join()
if errors: raise RuntimeError(f'Worker lỗi: {errors}. Xem runs/kaggle_seed*.log')
print(f'Hoàn tất hai seed trong {(time.time()-t0)/3600:.2f} giờ.')


In [ ]:
# Đóng gói kết quả Kaggle để chuyển sang Colab (giữ cả runs và partitions).
# Sau khi cell chạy xong, tải file từ Output/working rồi tải lên Colab theo hướng dẫn ở notebook seed 456.
archive = shutil.make_archive('/kaggle/working/kaggle_seeds_42_123','zip',root_dir='.',
                              base_dir='runs',)
print('Artifact:', archive)


## Bước tiếp theo

1. Lưu phiên bản Kaggle và tải `kaggle_seeds_42_123.zip` từ Output.
2. Mở `ViSEF_seed456_colab.ipynb`, tải archive lên Colab và chạy seed 456.
3. Sau khi có đủ ba seed, dùng script tổng hợp trong repository (`src/summarize_results.py`) trên bộ `runs` đã ghép để tạo bảng báo cáo. Notebook này chỉ chạy hai seed đầu, vì vậy không tính mean ± std cuối cùng.